# f-CBM — N24News multimodal pipelineCompanion notebook for **"Towards Faithful Multimodal Concept Bottleneck Models"**.End-to-end run of the multimodal pipeline on **N24News** with a `CLIP-base` backbone:dataset loading, black-box baseline, `f-CBM` training (KAN head + leakage loss),and the **KAN concept-to-class response curves** (Figure 6).**Setup** — see `CT-CBM/requirements.txt`; concept-annotated CSVs are expected under`data/datasets/N24News/` (see the repository README for how to obtain them).> Absolute cluster paths from the original run were rewritten to repository-relative `data/...` paths.> Stored outputs are from that run and are kept for reference.

In [ ]:
# --- bootstrap: make src/ importable and run from the repository root ---
import os, sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src" / "path_info.py").exists())
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)

In [ ]:
# Databricks/Colab only — locally, install from CT-CBM/requirements.txt
# !pip install adjustText

In [ ]:
import torch
import seaborn as sns
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset
import torch.nn.functional as F
from torchvision import transforms, datasets
from transformers import BlipModel, BlipProcessor
from tqdm import tqdm
import pandas as pd
import os
from sklearn.model_selection import StratifiedKFold
import numpy as np
import matplotlib.pyplot as plt
import glob
from PIL import Image
from sklearn.metrics import f1_score, accuracy_score, classification_report
import re
from functools import lru_cache
import multiprocessing
import mmap
import json

from transformers import BlipModel, BlipProcessor, CLIPModel, CLIPProcessor

from baseline_model import CustomClassifier, plot_history
from CBM_model import CBM_model, plot_concept_to_class_weights, compute_leakage_matrix
from data_N24_concepts import PATH_DATA_N24, load_data_N24

from CBM_pipeline import batch_size, device, num_workers, max_len, run_CBM

dataset_type = 'C3M'
backbone = 'clip'

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning, module="huggingface_hub.file_download")

os.environ["TOKENIZERS_PARALLELISM"] = "false" # to remove warnings about parallelism :
# huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
# To disable this warning, you can either:
# - Avoid using `tokenizers` before the fork if possible
# - Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)

# Import N24News dataset

In [ ]:
train_loader, val_loader, test_loader, class_dict, concept_list, concept_counts = load_data_N24(data_dir=PATH_DATA_N24, class_list=None, batch_size=batch_size, max_len=max_len, dataset_type=dataset_type, combine_type='combine', select_most_frequent=20)

In [ ]:
data_iter = iter(train_loader)
sample = next(data_iter)

print(sample.keys())

In [ ]:
import matplotlib.pyplot as plt
import textwrap

# Retrieve one batch from the DataLoader
batch = next(iter(val_loader))

index = 3
# Get the image, text, and label
image = batch['image'][index]
abstract = batch['text'][index]
label = batch['label'][index]

# Extract activated concepts
activated_concepts = []
for key in batch:
    if key.startswith("concept_") and batch[key][index] > 0:
        activated_concepts.append(key)

################################################################

def convert_back_to_image(pixel_values):

    print(f"\nProcessed tensor shape: {pixel_values.shape}")
    print(f"Tensor min value: {pixel_values.min().item()}")
    print(f"Tensor max value: {pixel_values.max().item()}")
    print(f"Tensor mean: {pixel_values.mean().item()}")

    # Denormalize
    mean = torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(3, 1, 1)
    std = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(3, 1, 1)

    denormalized = pixel_values * std + mean
    print(f"\nDenormalized min: {denormalized.min().item()}")
    print(f"Denormalized max: {denormalized.max().item()}")
    print(f"Denormalized mean: {denormalized.mean().item()}")

    # Clip and convert
    denormalized = torch.clamp(denormalized, 0, 1)
    image_np = denormalized.permute(1, 2, 0).numpy()

    print(f"\nNumpy array min: {image_np.min()}")
    print(f"Numpy array max: {image_np.max()}")
    print(f"Numpy array shape: {image_np.shape}")

    return image_np

############################################################

image_np = convert_back_to_image(image)

# Create a figure with two subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))
fig.suptitle(f"Label ID: {label.item()}, Label: {class_dict[label.item()]}", fontsize=16)

# Display the image on the left subplot
ax1.imshow(image_np)
ax1.axis("off")

# Prepare the abstract text
wrapped_abstract = textwrap.fill(f"Abstract: {abstract}", width=40)

# Prepare activated concepts text
if activated_concepts:
    concepts_text = textwrap.fill("Activated concepts:\n" + ", ".join(activated_concepts), width=40)
else:
    concepts_text = "No activated concepts"

# Combine abstract and concepts text
formatted_text = f"{wrapped_abstract}\n\n{concepts_text}"

# Display the formatted text on the right subplot
ax2.axis("off")
ax2.text(0.5, 0.5, formatted_text, fontsize=10, verticalalignment='center', wrap=True, ha='center')

plt.tight_layout()
plt.show()

## Black-box baseline

Reference CLIP classifier without a concept bottleneck.

> Not executed in this notebook — the baselines reported in the paper were run
> from `Exp_baseline.ipynb`. Kept here so the pipeline reads end to end.

In [ ]:
torch.cuda.empty_cache()

In [ ]:
from CBM_pipeline import run_baseline

history, model = run_baseline(backbone=backbone, num_epochs=10)

model.save_model(f"{PATH_DATA_N24}/models/{backbone}_baseline.pt")

## f-CBM — training (combine)

Joint training with the **KAN prediction head** (`kan_layer=True`) and the
**differentiable leakage loss** (`leakage_loss=True`) — the configuration of Section 4.

The concept subset below (39 concepts after filtering) is the one used for the
response curves; it is a subset of the 195 N24News concepts, chosen for legibility
of the figure.

In [ ]:
from CBM_pipeline import run_CBM
concept_scores = 'data/datasets/N24News/combine/cb_llm_annotation/outputs_concept_scoring/blue_checkpoints/clip/cavs/regression/sorted_macro_concepts_coverage_MJ_cb_llm_abs_all_LIG.pkl'

In [ ]:
select_concepts = ['concept_cryptocurrency', 'concept_championships', 'concept_athlete profiles', 'concept_cuisine trends', 'concept_restaurant developments', 'concept_chef profiles', 'concept_ingredients', 'concept_recipes', 'concept_food festivals', 'concept_culinary awards', 'concept_restaurants', 'concept_culinary institutes', 'concept_nutritional composition', 'concept_food science', 'concept_restaurant reviews', 'concept_chef interviews', 'concept_food origins', 'concept_album releases', 'concept_concert tours', 'concept_music festivals', 'concept_artist collaborations', 'concept_industry awards', 'concept_soundtrack releases', 'concept_record labels', 'concept_performance venues', 'concept_music publishers', 'concept_artist management', 'concept_album reviews', 'concept_artist interviews', 'concept_lyrical analysis', 'concept_commercial performance', 'concept_individual athletics','concept_digital products and services','concept_software applications', 'concept_data privacy','concept_social media',  'concept_career milestones', 'concept_tournaments', 'concept_athletic performance']

select_concepts_clean = [concept.replace('concept_', '') for concept in select_concepts]

history, model = run_CBM(dataset='N24', 
                dataset_type='CBLLM', 
                combine_type='combine', 
                backbone='clip', 
                concept_representation='importance', 
                num_epochs=10, 
                load=False, 
                plot=True, 
                leakage_loss=True, 
                #import_concept_list=concept_scores,
                #concept_level=12,
                select_concepts=select_concepts_clean,
                leakage_loss_activation='up', 
                kan_layer=True,
                loss_CBLLM='MSE',
                linear_layer_lr=1e-1,
                with_validation=True)

In [ ]:
from KAN import plot_concept_to_class_response_curves
import pickle

batch_size = 128
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_workers = multiprocessing.cpu_count()
max_len = 512

# import_concept_list = 'data/datasets/N24News/combine/cb_llm_annotation/outputs_concept_scoring/blue_checkpoints/clip/cavs/regression/sorted_macro_concepts_coverage_MJ_cb_llm_abs_all_LIG.pkl'
# concept_level = 12

# if(import_concept_list!=''):

#     _, ext = os.path.splitext(import_concept_list)  # ext like ".pkl" or ".json"

#     if ext.lower() in ('.pkl', '.pickle'):
#         with open(import_concept_list, 'rb') as f:
#             concept_ranking = pickle.load(f)

#             # concat lists from concept level = 0 until concept_level value
#             select_concepts = []
#             for level in range(concept_level+1):
#                 select_concepts += concept_ranking[level][0]
#                 coverage = concept_ranking[level][1]
#             print('import concept list')
#             print('coverage is', coverage, ' valid = ', coverage > 99)

#     elif ext.lower() == '.json':
#         with open(import_concept_list, 'r') as f:
#             concept_ranking = json.load(f)

#         # select first concepts until number concept_level
#         select_concepts = list(concept_ranking.keys())[:concept_level]
#         print('import concept list')

#     else:
#         raise ValueError(f"Unsupported concept list format: {ext}")

train_loader, val_loader, test_loader, class_dict, concept_list, concept_counts = load_data_N24(data_dir=PATH_DATA_N24, class_list=None, batch_size=batch_size, max_len=max_len, dataset_type='CBLLM', combine_type='combine', select_concepts=select_concepts_clean)

### KAN response curves (Figure 6)

The next cell **redefines** `plot_concept_to_class_response_curves` locally, overriding
the import from `KAN.py`. This local version is the one that produced Figure 6: it adds a
`list_to_plot` argument (the version in `KAN.py` picks 6 concepts *at random*), strips the
`concept_` prefix from labels, and sets the axis labels used in the paper
(*Concept Activation* / *KAN Response Curves*).

> This improvement was never back-ported into `KAN.py`.

In [ ]:
list_to_plot = ['concept_championships', 'concept_music festivals','concept_artist collaborations', 'concept_ingredients','concept_food festivals', 'concept_social media','concept_software applications']
list_to_plot_indices = [concept_list.index(concept) for concept in list_to_plot]

def plot_concept_to_class_response_curves(
    model,
    train_loader: torch.utils.data.DataLoader,
    feature_names: list,
    output_names: dict,
    n_points: int = 200,
    device: str = "cpu",
    list_to_plot=None,
    ):
    """
    Plots a grid showing the response curve of each input feature to every output neuron.

    Args:
        model: Trained KANLinear model
        train_loader: DataLoader yielding batches that are dicts; each feature_name
                      corresponds to a key in the batch, batch[feature_name] is (batch_size,)
        feature_names: List of feature names, order defines feature indices for the model
        output_names: Optional dict mapping output index -> name
        n_points: Number of points to evaluate each response curve
        device: Device where model lives ("cpu" / "cuda")
    """

    model.eval()
    model.to(device)

    in_features = model.in_features
    in_features_list = list(range(in_features))
    out_features = model.out_features
    out_features_list = list(range(out_features))

    if feature_names is None:
        raise ValueError("feature_names must be provided when using dict-style batches.")

    if len(feature_names) != in_features:
        raise ValueError(
            f"feature_names length ({len(feature_names)}) must match in_features ({in_features})"
        )

    # ------------------------------------------------------------------
    # 1. Compute per-feature min / max over the whole train_loader
    #    using batch[feature_name]
    # ------------------------------------------------------------------
    feature_mins = {fname: None for fname in feature_names}
    feature_maxs = {fname: None for fname in feature_names}

    with torch.no_grad():
        for batch in train_loader:
            # batch is expected to be a dict-like object
            for fname in feature_names:
                # batch[fname]: shape (batch_size,) or (batch_size, 1)
                vals = batch[fname]
                if isinstance(vals, torch.Tensor):
                    vals = vals.to(device).view(-1)
                else:
                    # if not tensor, convert
                    vals = torch.as_tensor(vals, device=device).view(-1)

                bmin = vals.min()
                bmax = vals.max()

                if feature_mins[fname] is None:
                    feature_mins[fname] = bmin
                    feature_maxs[fname] = bmax
                else:
                    feature_mins[fname] = torch.minimum(feature_mins[fname], bmin)
                    feature_maxs[fname] = torch.maximum(feature_maxs[fname], bmax)

    # Check that we saw at least one batch
    if any(v is None for v in feature_mins.values()):
        raise ValueError("train_loader appears to be empty or missing some feature keys.")

    # ------------------------------------------------------------------
    # 2. Possibly subsample outputs / inputs (as in your original code)
    # ------------------------------------------------------------------
    if len(out_features_list) > 4:
        out_features_selection = np.random.choice(out_features_list, 4, replace=False)
    else:
        out_features_selection = out_features_list

    if(list_to_plot==None):
        if len(in_features_list) > 6:
            in_features_selection = np.random.choice(in_features_list, 6, replace=False)
        else:
            in_features_selection = in_features_list
    else:
        in_features_selection = list_to_plot

    # ------------------------------------------------------------------
    # 3. Plot - Adjusted figsize to accommodate legend better
    # ------------------------------------------------------------------
    fig, axes = plt.subplots(
        1,
        len(out_features_selection),
        figsize=(20, 4.5),  # Reduced width and height for more compact layout
        sharex=True,
        squeeze=False,
    )

    # Store handles and labels for the shared legend
    handles, labels = None, None

    for out_idx_graph, out_idx in enumerate(out_features_selection):
        ax = axes[0, out_idx_graph]

        for _, feature_idx in enumerate(in_features_selection):
            feature_name = feature_names[feature_idx]

            feature_min = feature_mins[feature_name].item()
            feature_max = feature_maxs[feature_name].item()

            x_range = torch.linspace(feature_min, feature_max, n_points, device=device)

            # Response curve: (n_points, out_features)
            with torch.no_grad():
                resp = model.response_curve(
                    feature_idx=feature_idx,
                    x_values=x_range,
                    apply_scale=True,
                )

            y_curve = resp[:, out_idx].detach().cpu().numpy()
            ax.plot(x_range.cpu().numpy(), y_curve, label=feature_name.replace("concept_", ""))

        ax.set_title(output_names.get(out_idx, f"Output {out_idx}"), fontsize=20, pad=10)
        if out_idx_graph == 0:
            ax.set_ylabel("KAN Response Curves", fontsize=16)
        ax.grid(True, alpha=0.3)
        
        # Get handles and labels from the first subplot
        if handles is None:
            handles, labels = ax.get_legend_handles_labels()

    for ax in axes[-1]:
        ax.set_xlabel("Concept Activation", fontsize=16)
    
    # Create a single legend with better positioning
    fig.legend(handles, labels, loc='center left', bbox_to_anchor=(0.92, 0.5), 
               fontsize=10, frameon=True, fancybox=True, shadow=True)
    
    plt.tight_layout()
    plt.subplots_adjust(right=0.90)  # Less space on right - legend closer to plots
    plt.show()

plot_concept_to_class_response_curves(model.classifier, train_loader, concept_list, class_dict, list_to_plot=list_to_plot_indices)

In [ ]:
plot_concept_to_class_response_curves(model.classifier, train_loader, concept_list, class_dict)

In [ ]:
plot_concept_to_class_response_curves(model.classifier, train_loader, concept_list, class_dict)

## Variant — `concat` fusion

Same pipeline with `combine_type='concat'` instead of `combine`.

In [ ]:
run_CBM(dataset_type='CBLLM', combine_type='concat', backbone='clip', concept_representation='importance', num_epochs=5, select_most_frequent=5, load=False)

## Appendix — concept-loss weight (lambda)

Exploratory sweep over `lambda_XtoC`.

> ⚠️ This sweep was run on the **C3M** annotations with `select_most_frequent=10`,
> not on the `CBLLM` / 195-concept setting used for the results in the paper.
> It is kept as an exploration record and does **not** by itself justify the
> `lambda = 1` of Section 4.3.

In [ ]:
run_CBM(dataset_type=dataset_type, combine_type='combine', backbone=backbone, concept_representation='importance', num_epochs=10, select_most_frequent=10, load=False,lambda_XtoC=0.5)
run_CBM(dataset_type=dataset_type, combine_type='combine', backbone=backbone, concept_representation='importance', num_epochs=10, select_most_frequent=10, load=True,lambda_XtoC=0.5)

In [ ]:
run_CBM(dataset_type=dataset_type, combine_type='combine', backbone=backbone, concept_representation='importance', num_epochs=10, select_most_frequent=10, load=False,lambda_XtoC=5)
run_CBM(dataset_type=dataset_type, combine_type='combine', backbone=backbone, concept_representation='importance', num_epochs=10, select_most_frequent=10, load=True,lambda_XtoC=5)

In [ ]:
run_CBM(dataset_type=dataset_type, combine_type='combine', backbone=backbone, concept_representation='importance', num_epochs=10, select_most_frequent=10, load=False,lambda_XtoC=10)
run_CBM(dataset_type=dataset_type, combine_type='combine', backbone=backbone, concept_representation='importance', num_epochs=10, select_most_frequent=10, load=True,lambda_XtoC=10)